# 1. Construir CSVs con cada sujeto y todos sus mensajes unificados

In [1]:
# Importar librerías
import pandas as pd
import numpy as np
import json
import os
import csv

## Cargar CSVs originales

In [2]:
# conectar Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
# Rutas CSVs y JSONs
base = "/content/drive/MyDrive/tfg/corpusMentalRiskES/processed/"
paths = {
    "ED": {
        "csv": base + "ED/gold/gold_label.csv",
        "json": base + "ED/data/"
    },
    "Depression": {
        "csv": base + "Depress/gold/gold_label.csv",
        "json": base + "Depress/data/"
    },
    "Anxiety": {
        "csv": base + "Anxiety/gold/gold_label.csv",
        "json": base + "Anxiety/data/"
    }
}

# Leer los csv
df_anx = pd.read_csv(paths["Anxiety"]["csv"], sep="\t")
df_dep = pd.read_csv(paths["Depression"]["csv"], sep="\t")
df_ed = pd.read_csv(paths["ED"]["csv"], sep="\t")

## Generar DataFrames con mensajes unificados

In [3]:
def cargar_mensajes_usuarios(ruta_json_folder):
    """
    Devuelve un diccionario {user_id: [lista_de_mensajes]}
    a partir de los .json del trastorno.
    """
    mensajes = {}
    for filename in os.listdir(ruta_json_folder):
        if filename.endswith(".json"):
            user_id = filename.replace(".json", "")
            with open(os.path.join(ruta_json_folder, filename), "r", encoding="utf-8") as f:
                # Lista de diccionarios
                data = json.load(f)
                # Lista de mensajes no vacíos: tomamos m["message"] y descartamos textos vacíos o espacios
                textos = [m["message"] for m in data if m.get("message", "").strip() != ""]
                mensajes[user_id] = textos
    return mensajes

def preparar_dataframe_trastorno(df_labels, ruta_json_folder):
    """
    Construye un DataFrame con columnas:
    user_id, texto, bs
    """
    mensajes_por_usuario = cargar_mensajes_usuarios(ruta_json_folder)

    filas = []
    for _, row in df_labels.iterrows():
        user_id = str(row["nick"])
        bs = int(row["bs"])

        if user_id not in mensajes_por_usuario:
            continue  # por si hay algún usuario sin json

        # Obtener mensajes de cada usuario y unirlos
        msgs = mensajes_por_usuario[user_id]
        texto = "\n".join(msgs).strip()

        filas.append({
            "user_id": user_id,
            "texto": texto,
            "bs": bs
        })

    df_final = pd.DataFrame(filas)
    return df_final

In [13]:
df_anx_final = preparar_dataframe_trastorno(df_anx, paths["Anxiety"]["json"])
df_dep_final = preparar_dataframe_trastorno(df_dep, paths["Depression"]["json"])
df_ed_final = preparar_dataframe_trastorno(df_ed, paths["ED"]["json"])

display(df_anx_final)
print("\n")
display(df_dep_final)
print("\n")
display(df_ed_final)

,user_id,texto,bs
0,subject1,Buenos días a tod @s ! ! No sé si esta será la...,1
1,subject10,Ola soi nuebo como estn ?\nA ustds los entimdm...,0
2,subject100,De donde son .. Yo de perú\nSupongo q tb sigue...,1
3,subject101,Es lo que necesitas ?\nAaah pues perdón por no...,1
4,subject102,ya perdí una xD\nSiete seguidas no está mal .\...,1
...,...,...,...
495,subject95,Hola un saludo a todos\nMi nombre es oliver\nM...,1
496,subject96,Gracias la verdad que no estoy pasando buen mo...,1
497,subject97,Más aburrida que una ostra\nSugaramente un bue...,0
498,subject98,Hola ! ! Llevo un par de días leyendo algunos ...,1


,user_id,texto,bs
0,subject1,Bien ... técnicamente debería irme a dormir ca...,0
1,subject10,umm pues como te explico ... mal : ´ (\npues j...,1
2,subject100,"Hola , estoy realmente mal , no se que hacer c...",1
3,subject101,volvi me extrañaron signo de interrogación\ny ...,1
4,subject102,"¿ Cuanto tiempo duraron así ?\nsí , pero ya no...",1
...,...,...,...
494,subject95,"Buenas gente ! soy de Barcelona , alguien de p...",1
495,subject96,Como habéis pasado la noche\nMe tengo q acepta...,1
496,subject97,"Hola , no se casi del tema pero la verdad que ...",1
497,subject98,Yo no tengo ni ganas de vivir\nPues voy como v...,1


,user_id,texto,bs
0,subject1,hace cuanto lo crearon ?\naa es re nuevito\nso...,1
1,subject10,"Woww , muy bien lo que decis\nGracias por la i...",1
2,subject100,Me llamo Artemisa y me han diagnosticado BN\nT...,1
3,subject101,Jajaja bien y vos ? ?\nToca en donde dice ana ...,1
4,subject102,"hoola , me presento me llamo cayetana , soy en...",1
...,...,...,...
330,subject95,Cuantoa kilos sr puede bajar vomitando ?\nYa v...,1
331,subject96,"Holis grupo , porfis necesito perder 15 kilos ...",1
332,subject97,Quiero volver a bajar\nMe pasan licuados detox...,1
333,subject98,Hola . Vengo de una dieta cetogenica xon la qu...,1


# 2. Conjuntos para clasificación binaria

## 2.1. División train/val/test para BERT

Para entrenar y evaluar el modelo BERT, es necesario dividir el dataset en tres subconjuntos:

* **Entrenamiento** (70%)
* **Validación** (15%)
* **Test** (15%)

El conjunto de entrenamiento se utiliza para ajustar los pesos del modelo, el de validación para seleccionar hiperparámetros y evitar sobreajuste, y el de test para evaluar el rendimiento final.

Dado que el dataset está desbalanceado (especialmente en el caso del trastorno de ansiedad), la división se realiza mediante **estratificación**, utilizando la etiqueta bs como variable de estratificación. Esto asegura que la proporción entre las clases control y suffer se mantenga de forma similar en los tres subconjuntos.

La división se efectúa en dos pasos:
1.   Separar train del 30% restante.
2.   Dividir ese 30% en validación y test manteniendo también la estratificación.

In [ ]:
from sklearn.model_selection import train_test_split

def dividir_dataset(df, test_size=0.15, val_size=0.15, random_state=42):
    """
    Divide el dataset en train, val y test asegurando que la proporción
    entre clases (estratificación) se mantiene en cada subconjunto.
    """

    # 1. Train vs (Val + Test)
    df_train, df_temp = train_test_split(
        df,
        test_size = test_size + val_size,    # 0.30
        stratify = df["bs"],                 # mantiene proporción de clases
        random_state = random_state
    )

    # 2. Proporción val dentro del temporal (val + test)
    val_ratio = val_size / (test_size + val_size)

    # 3. Dividir temp → val y test
    df_val, df_test = train_test_split(
        df_temp,
        test_size = 1 - val_ratio,           # 0.5 (equivalente a 15% y 15%)
        stratify = df_temp["bs"],            # estratificación de nuevo
        random_state = random_state
    )

    return df_train, df_val, df_test

# Aplicar a cada trastorno
anx_train, anx_val, anx_test = dividir_dataset(df_anx_final)
dep_train, dep_val, dep_test = dividir_dataset(df_dep_final)
ed_train,  ed_val,  ed_test  = dividir_dataset(df_ed_final)

### Comprobar que los splits se hayan realizado correctamente manteniendo las distribuciones originales

In [ ]:
def imprimir_distribucion(df, nombre):
    print(f"\nDistribución de clases en {nombre}:")
    print(df["bs"].value_counts(normalize=False))      # valores absolutos
    print(df["bs"].value_counts(normalize=True).round(3))  # proporciones

imprimir_distribucion(anx_train, "Anxiety - Train")
imprimir_distribucion(anx_val,   "Anxiety - Validation")
imprimir_distribucion(anx_test,  "Anxiety - Test")

imprimir_distribucion(dep_train, "Depression - Train")
imprimir_distribucion(dep_val,   "Depression - Validation")
imprimir_distribucion(dep_test,  "Depression - Test")

imprimir_distribucion(ed_train, "ED - Train")
imprimir_distribucion(ed_val,   "ED - Validation")
imprimir_distribucion(ed_test,  "ED - Test")


Distribución de clases en Anxiety - Train:
bs
1    310
0     40
Name: count, dtype: int64
bs
1    0.886
0    0.114
Name: proportion, dtype: float64

Distribución de clases en Anxiety - Validation:
bs
1    67
0     8
Name: count, dtype: int64
bs
1    0.893
0    0.107
Name: proportion, dtype: float64

Distribución de clases en Anxiety - Test:
bs
1    66
0     9
Name: count, dtype: int64
bs
1    0.88
0    0.12
Name: proportion, dtype: float64

Distribución de clases en Depression - Train:
bs
1    233
0    116
Name: count, dtype: int64
bs
1    0.668
0    0.332
Name: proportion, dtype: float64

Distribución de clases en Depression - Validation:
bs
1    50
0    25
Name: count, dtype: int64
bs
1    0.667
0    0.333
Name: proportion, dtype: float64

Distribución de clases en Depression - Test:
bs
1    50
0    25
Name: count, dtype: int64
bs
1    0.667
0    0.333
Name: proportion, dtype: float64

Distribución de clases en ED - Train:
bs
0    134
1    100
Name: count, dtype: int64
bs
0    0.573

### Guardar los splits en CSVs

In [ ]:
output_path = "/content/drive/MyDrive/tfg/corpusMentalRiskES/splits/BERT/"
os.makedirs(output_path, exist_ok=True)

anx_train.to_csv(output_path + "anxiety_train.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
anx_val.to_csv(output_path + "anxiety_val.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
anx_test.to_csv(output_path + "anxiety_test.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)

dep_train.to_csv(output_path + "depression_train.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
dep_val.to_csv(output_path + "depression_val.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
dep_test.to_csv(output_path + "depression_test.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)

ed_train.to_csv(output_path + "ed_train.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
ed_val.to_csv(output_path + "ed_val.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
ed_test.to_csv(output_path + "ed_test.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)

### Análisis adicional de número de tokens

BERT tiene un límite de 512 tokens, por lo que es necesario ver si los textos superan este límite:





In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Calcular los tokens de TODOS los textos del dataset
def contar_tokens(df, tokenizer):
    longitudes = []
    for texto in df["texto"]:
        tokens = tokenizer.encode(texto, add_special_tokens=True)
        longitudes.append(len(tokens))
    return longitudes

tokens_anx = contar_tokens(df_anx_final, tokenizer)
tokens_dep = contar_tokens(df_dep_final, tokenizer)
tokens_ed  = contar_tokens(df_ed_final, tokenizer)

# Ver estadísticas generales
print("Ansiedad: media tokens =", np.mean(tokens_anx), ", máx tokens =", np.max(tokens_anx))
print("Depresión: media tokens =", np.mean(tokens_dep), ", máx tokens =", np.max(tokens_dep))
print("ED: media tokens =", np.mean(tokens_ed), ", máx tokens =", np.max(tokens_ed))

Token indices sequence length is longer than the specified maximum sequence length for this model (2567 > 512). Running this sequence through the model will result in indexing errors


Ansiedad: media tokens = 1042.98 , máx tokens = 6134
Depresión: media tokens = 820.2404809619238 , máx tokens = 6457
ED: media tokens = 799.5253731343283 , máx tokens = 12754


**Problema**: Muchos de los textos del dataset superan el límite máximo de 512 tokens que puede procesar BERT-base. Esto obliga a tomar una decisión sobre cómo manejar secuencias largas.

**Opciones disponibles**:

* **Truncación durante la tokenización**: Solo se conservan los primeros 512 tokens de cada texto. Esta solución es sencilla y eficiente, pero puede implicar pérdida de información.
* **Uso de un modelo Transformer de tipo long-range**: Estos modelos pueden procesar miles de tokens, evitando la necesidad de truncar. No obstante, su entrenamiento es más costoso en términos computacionales.

**Decisión**: En este TFG se emplearán ambos enfoques:

* BERT-base con truncación a 512 tokens, y
* un modelo long-range capaz de procesar secuencias completas.

Esto permitirá evaluar hasta qué punto la truncación afecta al rendimiento y si realmente compensa utilizar modelos long-range, que implican un mayor coste computacional.



## 2.2. Conjunto de evaluación para LLMs

En el caso de los LLMs no es necesario tener conjuntos train y val, ya que no hay entrenamiento. Por ello, se empleará como conjunto de evaluación el dataset completo.

Pero **en lugar de la versión *processed* del dataset se usará la versión *raw***, que incluye emojis que sí aceptan los LLMs.

In [4]:
# Rutas CSVs y JSONs de la versión raw
base_raw = "/content/drive/MyDrive/tfg/corpusMentalRiskES/raw/"
paths_raw = {
    "ED": {
        "csv": base_raw + "ED/gold/gold_label.csv",
        "json": base_raw + "ED/data/"
    },
    "Depression": {
        "csv": base_raw + "Depress/gold/gold_label.csv",
        "json": base_raw + "Depress/data/"
    },
    "Anxiety": {
        "csv": base_raw + "Anxiety/gold/gold_label.csv",
        "json": base_raw + "Anxiety/data/"
    }
}

# Leer los csv
df_raw_anx = pd.read_csv(paths_raw["Anxiety"]["csv"], sep="\t")
df_raw_dep = pd.read_csv(paths_raw["Depression"]["csv"], sep="\t")
df_raw_ed = pd.read_csv(paths_raw["ED"]["csv"], sep="\t")

In [7]:
# Generar dataframes con los mensajes unificados
df_raw_anx_final = preparar_dataframe_trastorno(df_raw_anx, paths_raw["Anxiety"]["json"])
df_raw_dep_final = preparar_dataframe_trastorno(df_raw_dep, paths_raw["Depression"]["json"])
df_raw_ed_final = preparar_dataframe_trastorno(df_raw_ed, paths_raw["ED"]["json"])

# display(df_raw_anx_final)
# print("\n")
# display(df_raw_dep_final)
# print("\n")
# display(df_raw_ed_final)

,user_id,texto,bs
0,subject1,Buenos días a tod @s ! ! No sé si esta será la...,1
1,subject10,Ola soi nuebo como estn ?\nA ustds los entimdm...,0
2,subject100,De donde son .. Yo de perú\nSupongo q tb sigue...,1
3,subject101,Es lo que necesitas ?\nAaah pues perdón por no...,1
4,subject102,ya perdí una xD\nSiete seguidas no está mal .\...,1
...,...,...,...
495,subject95,Hola un saludo a todos\nMi nombre es oliver\nM...,1
496,subject96,Gracias la verdad que no estoy pasando buen mo...,1
497,subject97,Más aburrida que una ostra\nSugaramente un bue...,0
498,subject98,Hola ! ! Llevo un par de días leyendo algunos ...,1


,user_id,texto,bs
0,subject1,Bien ... técnicamente debería irme a dormir 😅 ...,0
1,subject10,umm pues como te explico ... mal : ´ (\npues j...,1
2,subject100,"Hola , estoy realmente mal , no se que hacer c...",1
3,subject101,volvi me extrañaron ❓\ny tu pareja ❓\ntodo el ...,1
4,subject102,"¿ Cuanto tiempo duraron así ?\nsí , pero ya no...",1
...,...,...,...
494,subject95,"Buenas gente ! soy de Barcelona , alguien de p...",1
495,subject96,Como habéis pasado la noche\nMe tengo q acepta...,1
496,subject97,"Hola , no se casi del tema pero la verdad que ...",1
497,subject98,Yo no tengo ni ganas de vivir\nPues voy como v...,1


,user_id,texto,bs
0,subject1,hace cuanto lo crearon ?\naa es re nuevito\nso...,1
1,subject10,"Woww , muy bien lo que decis\nGracias por la i...",1
2,subject100,Me llamo Artemisa y me han diagnosticado BN\nT...,1
3,subject101,Jajaja bien y vos ? ?\nToca en donde dice ana ...,1
4,subject102,"hoola , me presento me llamo cayetana , soy en...",1
...,...,...,...
330,subject95,Cuantoa kilos sr puede bajar vomitando ?\nYa v...,1
331,subject96,"Holis grupo , porfis necesito perder 15 kilos ...",1
332,subject97,Quiero volver a bajar\nMe pasan licuados detox...,1
333,subject98,Hola . Vengo de una dieta cetogenica xon la qu...,1


In [10]:
output_path = "/content/drive/MyDrive/tfg/corpusMentalRiskES/splits/LLM/"
os.makedirs(output_path, exist_ok=True)

df_raw_anx_final.to_csv(output_path + "anxiety_llm.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
df_raw_dep_final.to_csv(output_path + "depression_llm.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
df_raw_ed_final.to_csv(output_path + "ed_llm.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)

# 3. Conjuntos para clasificación multiclase

En clasificació multiclase, en vez de entrenar un modelo por trastorno (ansiedad / depresión / ED), como en la clasificación binaria, ahora queremos un modelo que detecte cuál de los 3 trastornos sufre el usuario.

Para ello unificamos los dataframes de clasificación binaria en uno solo donde cada usuario pertenece a exactamente una clase. Por tanto, el dataset multiclase quedará así:

* Todos los usuarios de Anxiety → clase = 1
* Todos los usuarios de Depression → clase = 2
* Todos los usuarios de ED → clase = 3
* Los usuarios control dentro de cada carpeta siguen siendo “control”
→ clase = 0

In [ ]:
# Copia para no modificar los originales
df_anx_mc = df_anx_final.copy()
df_dep_mc = df_dep_final.copy()
df_ed_mc = df_ed_final.copy()

# Añadir columna con trastorno
df_anx_mc["trastorno"] = "anxiety"
df_dep_mc["trastorno"] = "depression"
df_ed_mc["trastorno"] = "ed"

# Crear la etiqueta multiclase
def asignar_etiqueta(row):
    if row["bs"] == 0:
        return 0  # control
    if row["trastorno"] == "anxiety":
        return 1
    if row["trastorno"] == "depression":
        return 2
    if row["trastorno"] == "ed":
        return 3

df_anx_mc["label_mc"] = df_anx_mc.apply(asignar_etiqueta, axis=1)
df_dep_mc["label_mc"] = df_dep_mc.apply(asignar_etiqueta, axis=1)
df_ed_mc["label_mc"] = df_ed_mc.apply(asignar_etiqueta, axis=1)

# Concatenar los tres dataset
df_multi = pd.concat([df_anx_mc, df_dep_mc, df_ed_mc], ignore_index=True)

print(df_multi.label_mc.value_counts())

El conjunto multiclase presenta un nivel moderado de desbalance entre categorías. La clase más numerosa es ansiedad (443 usuarios), seguida de control (415), depresión (333) y, finalmente, trastornos alimentarios (143), que constituye la clase minoritaria. Aunque el desbalance no es extremo, sí puede influir en el entrenamiento del modelo BERT multiclase, favoreciendo el aprendizaje de las clases más frecuentes y dificultando la detección de ED. Para mitigar este efecto, será importante emplear métricas equilibradas como el F1-macro y considerar técnicas como el uso de pesos de clase durante el entrenamiento.

## 3.1. División train/val/test para BERT

In [ ]:
# División estratificada
train_mc, temp_mc = train_test_split(
    df_multi,
    test_size=0.30,
    stratify=df_multi["label_mc"],
    random_state=42
)

val_mc, test_mc = train_test_split(
    temp_mc,
    test_size=0.50,
    stratify=temp_mc["label_mc"],
    random_state=42
)

In [ ]:
# Guardar en CSV
output_path = "/content/drive/MyDrive/tfg/corpusMentalRiskES/splits/BERT/"
os.makedirs(output_path, exist_ok=True)
train_mc.to_csv(output_path + "multiclass_train.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
val_mc.to_csv(output_path + "multiclass_val.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
test_mc.to_csv(output_path + "multiclass_test.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)

## 3.2. Conjunto de evaluación para LLMs

In [14]:
# Copias RAW
df_raw_anx_mc = df_raw_anx_final.copy()
df_raw_dep_mc = df_raw_dep_final.copy()
df_raw_ed_mc  = df_raw_ed_final.copy()

# Añadir trastorno
df_raw_anx_mc["trastorno"] = "anxiety"
df_raw_dep_mc["trastorno"] = "depression"
df_raw_ed_mc["trastorno"]  = "ed"

# Etiqueta multiclase
def asignar_etiqueta(row):
    if row["bs"] == 0:
        return 0
    if row["trastorno"] == "anxiety":
        return 1
    if row["trastorno"] == "depression":
        return 2
    if row["trastorno"] == "ed":
        return 3

for df in [df_raw_anx_mc, df_raw_dep_mc, df_raw_ed_mc]:
    df["label_mc"] = df.apply(asignar_etiqueta, axis=1)

# Dataset multiclase LLM (RAW)
df_raw_multi_llm = pd.concat([df_raw_anx_mc, df_raw_dep_mc, df_raw_ed_mc], ignore_index=True)

In [16]:
# Guardar el dataset completo adaptado a clasificacion multiclase
output_path = "/content/drive/MyDrive/tfg/corpusMentalRiskES/splits/LLM/"
os.makedirs(output_path, exist_ok=True)

df_raw_multi_llm.to_csv(output_path + "multiclass_llm.csv", index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)